<a href="https://colab.research.google.com/github/Srihari0804/Pneumonia-Detection/blob/main/Normal_Pneumonia_Xray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [2]:
import tensorflow as tf

In [3]:
#load train_ds
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
              '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray/train',
              labels="inferred",
               label_mode="categorical",
              class_names=["NORMAL","PNEUMONIA"], # Explicitly set class names to match your intended labels (0, 1 respectively).
              color_mode='rgb',
              batch_size=8,
              image_size=(229, 229),
              shuffle=True,
              seed= 42,
              validation_split=None,
              subset=None,
              interpolation='area',
              follow_links=False,
              crop_to_aspect_ratio=False,
              pad_to_aspect_ratio=True,
              data_format=None,
              verbose=True
          )

Found 5216 files belonging to 2 classes.


In [4]:
#load test_ds
test_ds,val_ds =  tf.keras.preprocessing.image_dataset_from_directory(
              '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray/test',
              labels="inferred",
              label_mode="categorical",
              class_names=["NORMAL","PNEUMONIA"], # Explicitly set class names to match your intended labels (0, 1, 2 respectively).
              color_mode='rgb',
              batch_size=8,
              image_size=(229, 229),
              shuffle=True,
              seed= 42,
              validation_split=0.3,
              subset="both",
              interpolation='area',
              follow_links=False,
              crop_to_aspect_ratio=False,
              pad_to_aspect_ratio=True,
              data_format=None,
              verbose=True
          )

Found 624 files belonging to 2 classes.
Using 437 files for training.
Using 187 files for validation.


In [5]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [6]:
#Model
i = tf.keras.Input(shape=(229,229,3))

preprocess_layer1 = tf.keras.layers.RandomContrast(0.3,seed=42)(i)
preprocess_layer2 = tf.keras.layers.GaussianNoise(stddev=0.3)(preprocess_layer1)
preprocess_layer3 = tf.keras.layers.RandomRotation(0.05,seed=42)(preprocess_layer2)
preprocess_layer4 = tf.keras.layers.RandomZoom(0.05)(preprocess_layer3)


rescale = tf.keras.applications.inception_resnet_v2.preprocess_input(
    preprocess_layer4, data_format=None)

base_model = tf.keras.applications.InceptionResNetV2(
                  include_top=False,
                  weights='imagenet',
                  input_tensor = rescale,
                  input_shape = (229,229,3),  #don't mention shape
                  pooling= "avg",
                  classifier_activation=None)

#Imp
base_model.trainable = True
base_model_output = base_model.output

custom_layer1 = tf.keras.layers.Dense(256 ,activation= "relu")(base_model_output) #relu activation
custom_layer2 = tf.keras.layers.Dense(256 ,activation= "relu")(custom_layer1)
layer3        = tf.keras.layers.Dense(128 ,activation= "relu")(custom_layer2)
layer4        = tf.keras.layers.Dense(64 ,activation= "relu")(layer3)

drop_out      = tf.keras.layers.Dropout(0.5)(layer4)
output_layer  = tf.keras.layers.Dense(2,activation="softmax")(drop_out)

model = tf.keras.Model(inputs=i, outputs=output_layer)

219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
#load saved weights:
model.load_weights("/content/Chest_Xray_model_weights.weights.h5")

In [9]:
model.compile(optimizer = tf.keras.optimizers.Adam(1e-5),
              loss= tf.keras.losses.BinaryCrossentropy(),
              metrics=[tf.keras.metrics.Recall(),tf.keras.metrics.Precision(),tf.keras.metrics.F1Score("macro")])

In [ ]:
checkpoint_filepath = '/content/retraining.model.keras'

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_f1_score',
    verbose=1,
    save_freq="epoch",
    mode='max',
    save_best_only=True)

In [ ]:
class_weights = {0:2.5, 1:1}

history = model.fit(train_ds, epochs=10,class_weight=class_weights,
                          callbacks = [tf.keras.callbacks.EarlyStopping(
                                monitor='val_f1_score',
                                min_delta=0.01,
                                patience=3,
                                verbose=1,
                                mode='max',
                                baseline=None,
                                restore_best_weights=True,
                                start_from_epoch=0), model_checkpoint_callback],
                    validation_data=val_ds)

Epoch 1/10
652/652 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - f1_score: 0.8371 - loss: 0.4288 - precision_4: 0.8648 - recall_4: 0.8648
Epoch 1: val_f1_score improved from -inf to 0.82314, saving model to /content/retraining.model.keras
652/652 ━━━━━━━━━━━━━━━━━━━━ 322s 343ms/step - f1_score: 0.8371 - loss: 0.4287 - precision_4: 0.8649 - recall_4: 0.8649 - val_f1_score: 0.8231 - val_loss: 0.4476 - val_precision_4: 0.8396 - val_recall_4: 0.8396
Epoch 2/10
652/652 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - f1_score: 0.9305 - loss: 0.2130 - precision_4: 0.9457 - recall_4: 0.9457
Epoch 2: val_f1_score improved from 0.82314 to 0.88021, saving model to /content/retraining.model.keras
652/652 ━━━━━━━━━━━━━━━━━━━━ 233s 357ms/step - f1_score: 0.9305 - loss: 0.2130 - precision_4: 0.9457 - recall_4: 0.9457 - val_f1_score: 0.8802 - val_loss: 0.2991 - val_precision_4: 0.8824 - val_recall_4: 0.8824
Epoch 3/10
652/652 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step - f1_score: 0.9504 - loss: 0.1500 - precision_4: 0.9615 - rec

In [ ]:
#save model weights, set the base model to Trainable, make training more hard

In [10]:
#Saving weights
model.save_weights("Chest_Xray_model_fineTuned_weights.weights.h5")

In [11]:
model.evaluate(test_ds)

55/55 ━━━━━━━━━━━━━━━━━━━━ 19s 100ms/step - f1_score: 0.4486 - loss: 0.6933 - precision: 0.4885 - recall: 0.4885


[0.7034639716148376,
 0.43478259444236755,
 0.43478259444236755,
 0.41309744119644165]

In [12]:
model.evaluate(train_ds)

652/652 ━━━━━━━━━━━━━━━━━━━━ 101s 69ms/step - f1_score: 0.4251 - loss: 0.7103 - precision: 0.4278 - recall: 0.4278


[0.711747944355011,
 0.41679447889328003,
 0.41679447889328003,
 0.4152737557888031]

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
#Checking saved model
saved_model = tf.keras.models.load_model("/content/retraining.model.keras")
saved_model.evaluate(test_ds)

55/55 ━━━━━━━━━━━━━━━━━━━━ 18s 92ms/step - f1_score: 0.8890 - loss: 0.3655 - precision_4: 0.8994 - recall_4: 0.8994


[0.33859553933143616,
 0.9038901329040527,
 0.9038901329040527,
 0.8919996023178101]

In [ ]:
saved_model.evaluate(train_ds)

652/652 ━━━━━━━━━━━━━━━━━━━━ 43s 66ms/step - f1_score: 0.9298 - loss: 0.1398 - precision_4: 0.9435 - recall_4: 0.9435


[0.130782350897789, 0.9482361674308777, 0.9482361674308777, 0.9361414313316345]

In [ ]:
saved_model.save_weights("weights_of_saved_model.weights.h5")